<a href="https://colab.research.google.com/github/davidmkidd/UK-Supermarket-Carbon-Emissions/blob/main/UKSmktComp_Retailer_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://evoviz.uk/wp-content/uploads/2026/04/Food_Divider_trans2.png">

# Retailer Comparison

The aim of this workbook is to produce graphic that compares the Scope 1 and 2 emissions of supermarket retailers.

Operational emissions consist of three portions (Fig. 1),

<br>

<img src="https://evoviz.uk/wp-content/uploads/2026/04/UKSmktComp_StackedComparison.png" width = 450>
<figcaption>Figure 1. Scope 1 and 2 Emissions Components.</figcaption>

<br>

1. Scope 1 emissions directly produced by the retailer

> $C_{scope1, location}$

2. Scope 2 Market-based emissions from purchased non-renewable power

> $C_{scope2, market}$

3. Scope 2 Location Residual emissions from purchased renewable power

> $C_{scope2,residual} = C_{scope2, location} - C_{scope2, market}$

> and

> $C_{scope2, residual} = C_{scope1and2, location} - C_{scope1and2, market}$

Scope 1, Scope 2, and Scope 1 and 2 location-based metrics scale with store area.

> $I_{scope} = C_{scope} / StoreArea$

To assess 'recent' emissions and reduce the impact of variation in published values between sources and recalculation, mean values are calculated for the 2020 - 2004 period.

Residual emissions are calculated from reported Scope 1, Scope 2 market metrics.

Problematic values are replaced with either residual emissions calculated from either Scope 1 and 2 location and market-based values, or from Scope 1 and Scope 1 and 2 location-based values.


## Set-up



In [ ]:
# Load libraries
library(dplyr)   # Data manipulation
library(ggplot2) # Graphing
library(repr)    # Graph size
options(repr.plot.width = 10, repr.plot.height = 8)
library(broom)   # Format model output
library(knitr)    # Format model output

# Download cleaned and summarised emmissions data
url <- "https://raw.githubusercontent.com/davidmkidd/UK-Supermarket-Carbon-Emissions/refs/heads/main/retailer_emissions_yr.csv"
download_path <- "/content/retailer_emissions_yr.csv"
download.file(url, destfile = download_path, mode = "wb")
emissions.yr <- read.csv("/content/retailer_emissions_yr.csv", header=TRUE, stringsAsFactors=FALSE)


# Download retailer data
url <- "https://raw.githubusercontent.com/davidmkidd/UK-Supermarket-Carbon-Emissions/refs/heads/main/retailer_data.csv"
download_path <- "/content/retailer_data.csv"
download.file(url, destfile = download_path, mode = "wb")
retailer.data <- read.csv("/content/retailer_data.csv", header=TRUE, stringsAsFactors=FALSE)


# Download retailer data
url <- "https://raw.githubusercontent.com/davidmkidd/UK-Supermarket-Carbon-Emissions/refs/heads/main/geolytix_retailpoints_v34_202412_summary.csv"
download_path <- "/content/geolytix_retailpoints_v34_202412_summary.csv"
download.file(url, destfile = download_path, mode = "wb")
retailer.store <- read.csv("/content/geolytix_retailpoints_v34_202412_summary.csv", header=TRUE, stringsAsFactors=FALSE)

# Make palette list
retailer.pal <- setNames(retailer.data$hex, retailer.data$retailer_code)
# Make code/name list
retailer.code <- setNames(retailer.data$retailer, retailer.data$retailer_code)

# Truncate year to 1s and 10s
emissions.yr$year2 <- emissions.yr$year - 2000

# Split reported absolute and intenstity values into seperate data frames as they will will be used differenty.

# Reported Absolute Values
emissions.yr.absolute <- emissions.yr %>%
   filter(kpi_type == "Measure")

# Reported Intensity Values
emissions.yr.intensity <- emissions.yr %>%
   filter(kpi_type == "Intensity")

# Calculate estimated intensity for absolute values
emissions.yr.absolute$intensity <- emissions.yr.absolute$value/emissions.yr.absolute$total_area

# Scope 1, Scope 2 Location, and Scope 2 Market

The first step is to obtain Scope 1, Scope 2 Market, and Scope 2 Location Residual values for retailers.

In [ ]:
# Scope 1 location-based emissions
scope1 <-  emissions.yr.absolute %>%
  filter(kpi == "Scope 1" & method == "Location") %>%
  select(retailer_code, retailer, kpi, year, intensity, value) %>%
  arrange(retailer_code, year)
scope1$metric <- "Scope 1"

In [ ]:
# Scope 2 location-based emissions
scope2.location <-  emissions.yr.absolute %>%
  filter(kpi == "Scope 2" & method == "Location") %>%
  select(retailer_code, retailer, kpi, year, intensity, value) %>%
  arrange(retailer_code, year)
scope2.location$metric <- "Scope 2 Location-based"

In [ ]:
# Scope 2 market-based emissions
scope2.market <-  emissions.yr.absolute %>%
  filter(kpi == "Scope 2" & method == "Market") %>%
  select(retailer_code, retailer, kpi, year, intensity, value) %>%
  arrange(retailer_code, year)
scope2.market$metric <- "Scope 2 Market-based"

Join all three datasets

In [ ]:
# Retailer years with Scope 2 location and market-based metrics
scope2.both <- merge(scope2.location, scope2.market,
                by.x=c("retailer_code", "year"),
                by.y=c("retailer_code", "year"))

scope2.both <- rename(scope2.both, location_intensity = intensity.x,
                 market_intensity = intensity.y,
                 location_value = value.x,
                 market_value = value.y,
                 retailer = retailer.x
                 )
scope2.both  <- select(scope2.both , -kpi.y, -kpi.x, -retailer.y, -metric.y, -metric.x)

# Records with scope 1 and both scope 2 records
three.values <- merge(scope2.both, scope1,
                            by.x=c("retailer_code","year"),
                            by.y=c("retailer_code","year"))

three.values <- three.values %>%
  rename(retailer = retailer.x, scope1_value = value)

nrow(three.values)

Calculate recent metric mean

In [ ]:
# Mean of recent retailer metrics
three.values.mean <- three.values %>%
  filter(year >= 2020 & year <= 2023) %>%
  summarise(
    scope1_value = mean(scope1_value),
    location_value = mean(location_value),
    market_value = mean(market_value),
    .by = c(retailer_code, retailer))

 names(three.values.mean)

Calculate Residual (offset)

In [ ]:
three.values.mean$offset_value <- three.values.mean$location_value - three.values.mean$market_value

names(three.values.mean)

Reformat to retailer_code, retailer, metric, value, then stack.



In [ ]:
# Scope 1
three.values.scope1 <- three.values.mean %>%
  select(retailer_code, retailer, scope1_value) %>%
  rename(value = scope1_value)
three.values.scope1$metric <- "Scope 1"
nrow(three.values.scope1)

In [ ]:
# Scope 2 Market
three.values.market <- three.values.mean %>%
  select(retailer_code, retailer, market_value) %>%
  rename(value = market_value)
three.values.market$metric <- "Scope 2 Market-based"
nrow(three.values.market)

In [ ]:
# Scope 2 Offset
three.values.offset <- three.values.mean %>%
  select(retailer_code, retailer, offset_value) %>%
  rename(value = offset_value)
three.values.offset$metric <- "Scope 2 Location Residual"
nrow(three.values.offset)

In [ ]:
# Stack
scope12 <- bind_rows(three.values.scope1, three.values.market, three.values.offset)
nrow(scope12)

Plot as stacked bar chart

In [ ]:
# Values are stacked in the chart in their factor order
# rev() reverses the factor vector
scope12$metric <- factor(scope12$metric,
      levels = rev(c("Scope 1", "Scope 2 Location-based",
      "Scope 2 Market-based", "Scope 2 Location Residual")))

In [ ]:
# Define custom color palette for metrics
metric_colors <- c(
  "Scope 1" = "#B22222",
  "Scope 2 Location-based" = "#EE7600",
  "Scope 2 Market-based" = "#EE2C2C",
  "Scope 2 Location Residual" = "#A2CD5A"
)

In [ ]:
# Plot Stacked Bar Chart
ggplot(scope12, aes(x = reorder(retailer_code, -value, sum), y = value/1000000, fill = metric)) +
  geom_bar(stat="identity") +
  labs(
    x = "Retailer",
    y = "Emissions Value",
    title = "Retailer Carbon Emissions by Scope (2020-2023)"
  ) +
  ylab(expression(paste("Million tCO"[2],"e"))) +
  scale_fill_manual(values = metric_colors) +
  theme_classic()

* Asda and Iceland do not publish all three scopes.

* Morrisons and M&S have negative scope 2 residual emissons!


> Morrisons only published Scope 2 Market-based values in its 2022 Annual Report. In this report, market-based values for three years are **larger** than location-based values, with location-based values noted as being,

>> *...taken from most recent invoice data which includes subsequent adjustments for rebilling; re-baselining of site inclusions/exclusions; and adjustments to the way data is apportioned across the year to ensure ongoing consistency.*

> Morrison's recent mean  Scope 2 location-based value is therefore not trusted.

>> Marks and Spencer's 2024 Annual Report recaculates 2023 Scope 2 market-based emissions as approx 66% greater than previously reported.

> Marks and Spencer recent Scope 2 location-based value is therefore not trusted.

Data for Asda, Iceland, Morrisons and Marks and Specncer may extracted from from reported combined Scope 1 and 2 emissions.




In [ ]:
# Exclude M&S and Morrisons
scope12 <- scope12 %>%
  filter(!(retailer_code == "MORR" | retailer_code == "M&S"))

# Combined Scope 1 and 2 values

Get emissions.

In [ ]:
# Scope 1 and 2 location-based emissions
scope12.location <-  emissions.yr.absolute %>%
  filter(kpi == "Scope 1 and 2" & method == "Location") %>%
  select(retailer_code, retailer, kpi, year, intensity, value) %>%
  arrange(retailer_code, year)
scope12.location$metric <- "Scope 1 and 2 Location-based"
nrow(scope12.location)

In [ ]:
# Scope 1 and 2 market-based emissions
scope12.market <-  emissions.yr.absolute %>%
  filter(kpi == "Scope 1 and 2" & method == "Market") %>%
  select(retailer_code, retailer, kpi, year, intensity, value) %>%
  arrange(retailer_code, year)
scope12.market$metric <- "Scope 1 and 2 Market-based"

nrow(scope12.market)

Join to get records with Scope 1 and Scope 1 and 2 location and market-based values.

In [ ]:
# Retailer years with Scope 1 and 2 location and market-based metrics
scope12.both <- merge(scope12.location, scope12.market,
                by.x=c("retailer_code", "year"),
                by.y=c("retailer_code", "year"))

scope12.both <- rename(scope12.both, location_intensity = intensity.x,
                 market_intensity = intensity.y,
                 location_value = value.x,
                 market_value = value.y,
                 retailer = retailer.x
                 )

scope12.both  <- select(scope12.both , -kpi.y, -kpi.x, -retailer.y)
nrow(scope12.both)

In [ ]:
# Records with scope 1 and scope 2 records
both.values <- merge(scope12.both, scope1,
                            by.x=c("retailer_code","year"),
                            by.y=c("retailer_code","year"))

both.values <- both.values %>%
  rename(retailer = retailer.x, scope1_value = value)

nrow(both.values)

Calculate recent mean values

In [ ]:
# Mean of Scope 1 and 2
scope12.both.mean <- both.values %>%
  filter(year >= 2020 & year <= 2023) %>%
  summarise(
    scope1_value = mean(scope1_value),
    location_value = mean(location_value),
    market_value = mean(market_value),
    .by = c(retailer_code, retailer))

Calculate residual (offset) and Scope 2 market-based

In [ ]:
scope12.both.mean$offset <- scope12.both.mean$location_value - scope12.both.mean$market_value
scope12.both.mean$s2_market_value <- scope12.both.mean$market_value - scope12.both.mean$scope1_value

Reformat to retailer_code, retailer, metric, value, then stack.

In [ ]:
# Scope 1
scope12.both.scope1 <- scope12.both.mean %>%
  select(retailer_code, retailer, scope1_value)
scope12.both.scope1$metric <- "Scope 1"
scope12.both.scope1 <- scope12.both.scope1 %>%
  rename(value = scope1_value)
nrow(scope12.both.scope1)

# Scope 2 Market
scope12.both.market <- scope12.both.mean %>%
  select(retailer_code, retailer, s2_market_value)
scope12.both.market$metric <- "Scope 2 Market-based"
scope12.both.market <- scope12.both.market %>%
  rename(value = s2_market_value)
nrow(scope12.both.market)

# Scope 2 Residual
scope12.both.offset <- scope12.both.mean %>%
  select(retailer_code, retailer, offset)
scope12.both.offset$metric <- "Scope 2 Location Residual"
scope12.both.offset <- scope12.both.offset %>%
  rename(value = offset)
nrow(scope12.both.offset)


In [ ]:
# Stack
scope1and2 <- bind_rows(scope12.both.scope1, scope12.both.market, scope12.both.offset)
nrow(scope1and2)

Stacked bar chart.

In [ ]:
scope1and2$metric <- factor(scope1and2$metric,
      levels = rev(c("Scope 1","Scope 1 and 2 Market-based", "Scope 2 Location-based",
      "Scope 2 Market-based", "Scope 2 Location Residual")))

# Define custom color palette for metrics
metric_colors <- c(
  "Scope 1" = "#B22222",
  "Scope 1 and 2 Location-based" = "#8B2E16",
  "Scope 2 Location-based" = "#EE7600",
  "Scope 2 Market-based" = "#EE2C2C",
  "Scope 2 Location Residual" = "#A2CD5A"
)

ggplot(scope1and2, aes(x = reorder(retailer_code, -value, sum), y = value/1000000, fill = metric)) +
  geom_bar(stat="identity") +
  labs(
    x = "Retailer",
    y = "Emissions Value",
    title = "Retailer Carbon Emissions by Scope (2020-2023)"
  ) +
  ylab(expression(paste("Million tCO"[2],"e"))) +
  scale_fill_manual(values = metric_colors) +
  theme_classic()

* Tesco has negative Scope 2 Market-based emissions!

* Morrisons has negative Scope 2 location-based emissions!

* M&S is reasonable.

Append ICE and M&S to scope12.

In [ ]:
ice.ocado <- scope1and2 %>%
   filter(retailer_code == "M&S" | retailer_code == "ICE")
ice.ocado


In [ ]:
data <- bind_rows(scope12, ice.ocado)

# Scope 1 and Scope 2 location

Asda and Morrisons have Scope 1 and Scope 2 location values

In [ ]:
# Scope 1
ret.scope1 <- scope1 %>%
  filter((year >= 2020 & year <= 2023) & (retailer_code == "ASDA" | retailer_code == "MORR") ) %>%
  summarise(
    scope1_value = mean(value),
    .by = c(retailer_code, retailer)) %>%
      rename(value = scope1_value)
ret.scope1$metric <- "Scope 1"


In [ ]:
# Scope 2 location-based
ret.location <- scope2.location %>%
  filter((year >= 2020 & year <= 2023) & (retailer_code == "ASDA" | retailer_code == "MORR")) %>%
  summarise(
    location_value = mean(value),
    .by = c(retailer_code, retailer)) %>%
  rename(value = location_value)
ret.location$metric <- "Scope 2 Location-based"

In [ ]:
# Append
data <- bind_rows(data, ret.scope1, ret.location)

# Summarise Retailer Store Area for 2020-2023

In [ ]:
# Summary of retailer for same period

store.stats <- retailer.store %>%
  filter(year >= 2020 & year <= 2023 & retailer %in% retailer.data$retailer) %>%
  summarise(
    total_store = mean(total_store),
    total_area = mean(total_area),
    .by = c(retailer))

In [ ]:
# Join to data
data <- merge(data, store.stats, by = "retailer")
nrow(data)

# Calculate and Plot Intensity

In [ ]:
# Intensity
data$intensity <- data$value / data$total_area

Create ordering_intensity field as sum of Scope 1, Scope 2 Market-based, and Scope 2 Location-based intensities

In [ ]:
# Create a temporary column for ordering based on the sum of specified metrics
  data.ordered <- data %>%
  mutate(ordering_intensity = ifelse(
    metric %in% c("Scope 1", "Scope 2 Market-based", "Scope 2 Location-based", "Scope 2 Location Residual") ,
    intensity, 0
  ))
# Set factor order
  data.ordered$metric <- factor(data.ordered$metric,
      levels = rev(c("Scope 1",
      "Scope 2 Market-based",
      "Scope 2 Location-based",
      "Scope 2 Location Residual")))

In [ ]:
# Plot with ordering
ggplot(data.ordered, aes(x = reorder(retailer_code, -ordering_intensity, sum), y = intensity, fill = metric)) +
  geom_bar(stat="identity") +
  labs(
    x = "Retailer",
    y = "Emissions Value",
    title = "Retailer Carbon Emission Intensity by Scope (2020-2023)"
  ) +
  ylab(expression(paste("tCO"[2],"e/m"^2))) +
  scale_fill_manual(values = metric_colors) +
  theme_classic()

This is the final UK Supermarker Retailer Emissions comparison graph.

Retailers with stores are ordered by their total Scope 1 and 2 location-based emissions. Scope 2 location-based values could not be separated into market-based emissions and Scope 2 location residuals discounted through the purchase of renewable energy.

By intensity scaled to total store area between 2021 and 2023:

* Tesco has the highest total intensity signisficatly greater than other retailers.

* Total intensity of low cost retailers is lower than other retailers.

* Marks and Spencer and Sainsburys have the highest Scope 2 market intensity for emissions from purchased power.

* Other retailers mostly or entirely purchase renewable energy, so have no Scope 2 market-based emissions.

* Morrison and Asda Sope 2 emissions could not be split into market  and redisual emissions.


So the results are in, and the 'UK Supermarket Carbon Emittor 2021-2023 Awards are as follows:

* The **Lowest Emittor** award goes to Lidl, for an exceptional performance with an operational emission intensity over five times less than the highest emittor.

* The **Highest Emittor** award goes to Tesco.

* Ocado was forced to exit the comptetition as it does not have any stores.

* Iceland's second position is at odds with Zuke 2025 which reported it has the highest intensity by revenue.



# References
Zuke, E. (2025). Iceland most carbonintense supermarket. Grocer, , 5. Retrieved from https://www.proquest.com/trade-journals/iceland-most-carbonintense-supermarket/docview/3256991965/se-2

---

[Main Page](https://colab.research.google.com/drive/1f8a0pXfF9PqCujiwjf4TO4-k7ezt-6b3?usp=sharing)

---
